# 🔍 Guía y Validación: ANTES vs DESPUÉS de la Limpieza
## Tablas: `coupons`, `categories` e `inventory`

Bienvenida a este notebook tutorial. Aquí aprenderás la **técnica profesional de validación de calidad de datos en PySpark**:

### 🧠 La Técnica de Validación "Lado a Lado":
1. **No sobreescribir la columna original inmediatamente**: En el limpiador creamos columnas nuevas con el sufijo `_clean` (ej: `code_clean`, `discount_type_clean`).
2. **Poner original y limpio lado a lado**: Con `.select("col_original", "col_clean")` para ver la transformación con tus propios ojos.
3. **Filtrar solo las filas que sufrieron cambios**: Con `df.filter(col("original") != col("clean"))` para no perder tiempo mirando filas idénticas.
4. **Medir el impacto**: Contar cuántos registros cambiaron y cuántos fueron marcados como `UNKNOWN` o `null`.

---

## ⚙️ Celda 1 — Conexión y Carga de Datos Crudos (Bronze)

In [ ]:
# Inicialización universal de la sesión de Spark (funciona en Databricks y en local con Databricks Connect)
try:
    spark
except NameError:
    try:
        from databricks.connect import DatabricksSession
        spark = DatabricksSession.builder.getOrCreate()
    except Exception:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.appName("ValidacionNuevasTablas").getOrCreate()

# Rutas al contenedor Bronze en Azure Data Lake
storage_account = "stecommercedirty01"  # Tu storage account
container = "bronze"

path_coupons = f"abfss://{container}@{storage_account}.dfs.core.windows.net/coupons/"
path_categories = f"abfss://{container}@{storage_account}.dfs.core.windows.net/categories/"
path_inventory = f"abfss://{container}@{storage_account}.dfs.core.windows.net/inventory/"

# Lectura en modo multilínea de los JSONs crudos
df_coupons_raw = spark.read.option("multiLine", True).json(path_coupons)
df_categories_raw = spark.read.option("multiLine", True).json(path_categories)
df_inventory_raw = spark.read.option("multiLine", True).json(path_inventory)

print(f"✅ Coupons cargados:    {df_coupons_raw.count()} filas")
print(f"✅ Categories cargadas: {df_categories_raw.count()} filas")
print(f"✅ Inventory cargado:   {df_inventory_raw.count()} filas")


---
## 🎟️ SECCIÓN 1 — Validación de `coupons`

Vamos a aplicar `run_coupon_cleaning()` y comparar:
- `code` vs `code_clean`
- `discount_type` vs `discount_type_clean`
- `expires_at` vs `expires_at_clean`
- Esquema y tipos numéricos casteados

In [ ]:
from pyspark.sql import functions as F
from transformacion.cleaning.coupon_cleaner import run_coupon_cleaning

# 1. Aplicamos la limpieza completa de cupones
df_coupons_clean = run_coupon_cleaning(df_coupons_raw, spark)

# 2. COMPARACIÓN LADO A LADO: Código del cupón y tipo de descuento
print("📋 1. Comparación Lado a Lado: Código y Tipo de Descuento (Muestra de 20)")
df_coupons_clean.select(
    "code", "code_clean",
    "discount_type", "discount_type_clean",
    "expires_at", "expires_at_clean"
).show(20, truncate=False)


In [ ]:
# 3. FILTRAR FILAS DONDE EL TIPO DE DESCUENTO CAMBIÓ O SE NORMALIZÓ
print("🔍 2. Registros donde discount_type fue corregido o estandarizado:")
df_coupons_clean.filter(
    F.col("discount_type") != F.col("discount_type_clean")
).select(
    "code", "discount_type", "discount_type_clean"
).show(20, truncate=False)

# 4. DETECTAR DESCUENTOS QUE NO PERTENECÍAN AL CATÁLOGO (quedaron UNKNOWN)
unknown_discounts = df_coupons_clean.filter(F.col("discount_type_clean") == "UNKNOWN")
print(f"⚠️  Descuentos inválidos marcados como UNKNOWN: {unknown_discounts.count()}")
unknown_discounts.select("id", "code", "discount_type", "discount_type_clean").show(truncate=False)


In [ ]:
# 5. VERIFICACIÓN DE TIPOS DE DATOS (SCHEMA DESPUÉS DE LA LIMPIEZA)
print("📋 Tipos de datos en Coupons después de safe_cast:")
df_coupons_clean.select(
    "id", "discount_value", "min_order", "max_uses", "used_count", "is_active", "expires_at_clean"
).printSchema()


---
## 🏷️ SECCIÓN 2 — Validación de `categories`

En categorías aprenderás cómo validar:
- `name` vs `name_clean`: verificar que respetó el símbolo `&` en *Food & Beverages* y capitalizó en *Title Case*.
- `description` vs `description_clean`: verificar que los nulos o vacíos se sustituyeron por `'No description'`.
- `created_at` vs `created_at_clean`: verificar cómo `'bad-date'` fue controlado de forma segura.

In [ ]:
from transformacion.cleaning.category_cleaner import run_category_cleaning

# 1. Ejecutar limpieza de categorías
df_categories_clean = run_category_cleaning(df_categories_raw, spark)

# 2. Comparación Lado a Lado de todas las categorías
print("📋 Comparación Antes vs Después en Categories:")
df_categories_clean.select(
    "id",
    "name", "name_clean",
    "description", "description_clean",
    "is_active",
    "created_at", "created_at_clean"
).show(truncate=False)


In [ ]:
# 3. VERIFICAR EL TRATAMIENTO DE FECHAS CORRUPTAS ('bad-date')
print("🔍 Registros con fechas corruptas en created_at:")
df_categories_clean.filter(
    F.col("created_at") == "bad-date"
).select(
    "id", "name", "created_at", "created_at_clean"
).show(truncate=False)
print("Nota cómo 'bad-date' quedó limpio como 'null' sin romper la ejecución.")


---
## 📦 SECCIÓN 3 — Validación de `inventory`

En inventario comparamos:
- `warehouse` vs `warehouse_clean`: eliminación de espacios adelante/atrás y mayúsculas desordenadas.
- `status` vs `status_clean`: unificación de `'in stock'`, `'LOW_STOCK'`, `'discontinued'`, etc. a *snake_case*.
- `last_restock` vs `last_restock_clean`: estandarización a formato de fecha ISO.

In [ ]:
from transformacion.cleaning.inventory_cleaner import run_inventory_cleaning

# 1. Ejecutar limpieza de inventario
df_inventory_clean = run_inventory_cleaning(df_inventory_raw, spark)

# 2. Comparación Lado a Lado: Almacén y Estado de Stock
print("📋 Comparación Antes vs Después en Almacenes y Estados:")
df_inventory_clean.select(
    "warehouse", "warehouse_clean",
    "status", "status_clean",
    "last_restock", "last_restock_clean"
).show(25, truncate=False)


In [ ]:
# 3. RESUMEN DE CAMBIOS EN 'status': ¿Cómo quedaron distribuidos los estados limpios?
print("📊 Distribución de status_clean después de estandarizar:")
df_inventory_clean.groupBy("status_clean").count().orderBy(F.col("count").desc()).show(truncate=False)

# 4. FILTRAR FILAS DONDE EL ALMACÉN TENÍA ESPACIOS EXTRA O ERA NULO/VACÍO
almacenes_corregidos = df_inventory_clean.filter(
    F.col("warehouse") != F.col("warehouse_clean")
)
print(f"✏️  Almacenes corregidos (espacios o vacíos): {almacenes_corregidos.count()}")
almacenes_corregidos.select("warehouse", "warehouse_clean").distinct().show(15, truncate=False)


---
## 🚀 SECCIÓN 4 — Validación del Pipeline Completo Orquestado

El orquestador `run_transformacion` ejecuta todas las tablas en un único proceso. Aquí comprobamos que devuelve un diccionario con todas las entidades transformadas listas para Silver Layer.

In [ ]:
from transformacion.pipeline_transformacion import run_transformacion

# Diccionario de tablas crudas de entrada
dataframes_crudos = {
    "coupons": df_coupons_raw,
    "categories": df_categories_raw,
    "inventory": df_inventory_raw,
}

# Ejecución centralizada
dataframes_limpios = run_transformacion(dataframes_crudos, spark)

print("🎉 ¡Pipeline transformado exitosamente!")
for entidad, df_res in dataframes_limpios.items():
    print(f"  • Entidad '{entidad}': {df_res.count()} registros procesados con columnas de auditoría ({[c for c in df_res.columns if 'audit' in c or 'ingestion' in c]})")
